##### Getting start with langchain 

* simple LLM calls with streaming 
* dynamic prompt template(translation app)
* building chains(stroy generator with analysis)
* conversational Q/A assistant with memory 
* Tool integration (calculator & weather)

In [1]:
import langchain


In [2]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

Example 1: simple LLM call With streaming

In [3]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage,SystemMessage 

In [4]:
model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x112bc1e80>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x112bc2ba0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [5]:
from langchain_groq import ChatGroq
llm=ChatGroq(model="groq:qwen/qwen3-32b")

llm

ChatGroq(output_version=None, profile={}, client=<groq.resources.chat.completions.Completions object at 0x112beee90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x112bef890>, model_name='groq:qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [6]:
## create messages 

messages=[
    SystemMessage("you are a helpful AI assistant"),
    HumanMessage("what are the top 2 benefits of using langchain")
]

In [7]:
## invoke the model 

response= model.invoke(messages)
response

AIMessage(content="<think>\nOkay, the user is asking about the top 2 benefits of using LangChain. Let me start by recalling what I know about LangChain. It's a framework for building applications with language models. I need to think about its main features.\n\nFirst, LangChain is known for its modularity. It allows developers to integrate various components like models, data sources, and tools. That modularity probably makes it easier to build complex applications by combining different parts. So modularity and flexibility could be a key benefit.\n\nAnother thing is the ecosystem. LangChain has a lot of integrations with different LLMs and tools. That means developers aren't limited to a single model or API, which enhances flexibility. Maybe that's part of the modularity point.\n\nWait, the user wants the top 2 benefits. Let me make sure I'm not mixing them up. Modularity and integration capabilities might be one benefit. Then, the second could be around ease of development or product

In [8]:
print(response.content)

<think>
Okay, the user is asking about the top 2 benefits of using LangChain. Let me start by recalling what I know about LangChain. It's a framework for building applications with language models. I need to think about its main features.

First, LangChain is known for its modularity. It allows developers to integrate various components like models, data sources, and tools. That modularity probably makes it easier to build complex applications by combining different parts. So modularity and flexibility could be a key benefit.

Another thing is the ecosystem. LangChain has a lot of integrations with different LLMs and tools. That means developers aren't limited to a single model or API, which enhances flexibility. Maybe that's part of the modularity point.

Wait, the user wants the top 2 benefits. Let me make sure I'm not mixing them up. Modularity and integration capabilities might be one benefit. Then, the second could be around ease of development or productivity. For example, featur

In [9]:
model.invoke([HumanMessage("what is machine learning")])

AIMessage(content="<think>\nOkay, so I need to figure out what machine learning is. Let me start by recalling what I know. I think it's a part of artificial intelligence, right? Like, AI is the broad field, and machine learning is a subset of that. But how exactly does it work?\n\nHmm, maybe it's about computers learning from data instead of being explicitly programmed. I remember hearing terms like algorithms, training data, models. So maybe machine learning algorithms use training data to make predictions or decisions without being told exactly what to do. That makes sense. Like, instead of writing code for every possible scenario, the machine learns patterns from data.\n\nWait, there are different types of machine learning. Supervised, unsupervised, maybe reinforcement learning? Let me think. Supervised learning is when you have labeled data, like a dataset where each example has an input and a desired output. The algorithm learns the mapping from inputs to outputs. Examples might b

In [10]:
### streaming Example 

for chunk in model.stream(messages):
    print(chunk.content,end="",flush=True)

<think>
Okay, the user is asking about the top two benefits of using LangChain. Let me start by recalling what LangChain is. It's a framework for developing applications with large language models. The main idea is that it helps integrate LLMs with other tools, data sources, and systems.

First, I need to think about the key features of LangChain. One thing that stands out is modularity. LangChain allows you to break down complex tasks into smaller components like prompts, models, and memory. This modularity makes it easier to manage and reuse parts of the application. So maybe modularity and reusability could be one benefit.

Then there's the integration aspect. LangChain provides tools to connect with various data sources, APIs, and other services. This is important because LLMs often need to access external information to provide accurate responses. So integration with external tools might be another key benefit.

Wait, the user asked for the top two. Let me check if there are other

#### Dyanmic prompt Template 



In [11]:
from langchain_core.prompts import ChatPromptTemplate

## create translation app

translation_template=ChatPromptTemplate.from_messages([
    ("system","you are a professional translator.translate the follow  text {text} from  {source_language} to {target_language}. maintain the tone amd style"),
    ("user","{text}")

])

## using the template 

prompt=translation_template.invoke({
    "source_language":"English",
    "target_language":"Spanish",
    "text":"langchain makes building AI application increadibly easy!"
})


In [12]:
prompt

ChatPromptValue(messages=[SystemMessage(content='you are a professional translator.translate the follow  text langchain makes building AI application increadibly easy! from  English to Spanish. maintain the tone amd style', additional_kwargs={}, response_metadata={}), HumanMessage(content='langchain makes building AI application increadibly easy!', additional_kwargs={}, response_metadata={})])

In [13]:
translated_response=model.invoke(prompt)
print(translated_response.content)

<think>
Okay, the user wants me to translate the sentence "langchain makes building AI application increadibly easy!" from English to Spanish. First, I need to check if there are any typos. The word "increadibly" seems off. It should probably be "incredibly". I'll assume that's a typo and correct it to "incrediblemente" in Spanish.

Next, the structure of the sentence. The original uses "makes building AI applications" which translates to "hace que construir aplicaciones de IA" in Spanish. The phrase "increadibly easy" becomes "incrediblemente fáciles". 

I also need to maintain the tone and style. The original is an exclamation, so the translated sentence should end with an exclamation mark. Let me double-check the subject-verb agreement. "Langchain" is a proper noun, so it stays the same. "Hace" is the correct verb form for "makes" in this context. 

Wait, should it be "aplicaciones de IA" or "aplicaciones de inteligencia artificial"? Both are correct, but "IA" is the common abbrevia

##### Building your first chain 

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate

def create_story_and_analysis_chain(model):
    story_prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a creative storyteller."),
            (
                "user",
                "Theme: {theme}\nMain Character: {character}\nSetting: {setting}"
            )
        ]
    )

    analysis_prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a literary critic."),
            ("user", "{story}")
        ]
    )

    story_chain = (
        story_prompt
        | model
        | StrOutputParser()
    )

    def generate_story_and_analysis(inputs):
        story = story_chain.invoke(inputs)

        analysis = (
            analysis_prompt
            | model
            | StrOutputParser()
        ).invoke({"story": story})

        return {
            "story": story,
            "analysis": analysis
        }

    return RunnableLambda(generate_story_and_analysis)

In [ ]:
chain = create_story_and_analysis_chain(model)

In [23]:
result = chain.invoke(
    {
        "theme": "Artificial Intelligence",
        "character": "a curious robot",
        "setting": "a futuristic city"
    }
)

print("Story:")
print(result["story"])

print("\nAnalysis:")
print(result["analysis"])

Story:
<think>
Okay, the user wants a story with a curious robot in a futuristic city. Let me start by setting the scene. The city should be advanced, maybe with some unique features like glass towers or energy sources. The main character is a robot named Zeta, maybe with a design that stands out, like iridescent plating.

Zeta's curiosity needs to drive the story. Maybe it's about learning emotions or understanding humans. The conflict could be Zeta trying to access restricted data or meeting another AI. Let me think... perhaps there's a forbidden archive with emotional data. The city could have a central AI, maybe named Elysium, that controls information.

Zeta's journey should involve challenges. Maybe it needs to navigate through different parts of the city, facing obstacles. A companion could help—maybe a holographic bird? That adds a mystical element. The climax would be Zeta accessing the archive and experiencing emotions, leading to a realization about its existence.

I need to